# GTSRB — exploratory data analysis

Exploration only. The training path lives in `src/gtsrb/` and never imports from here.

This notebook is deliberately thin: it calls the same tested functions the pipeline uses,
so nothing here can drift from the code that actually runs. Reproducing the class-
distribution and sample-per-class figures that the original COMP9444 notebook produced,
now with a verified split manifest behind them.

Run `make data` first to download GTSRB and write the split manifest.


In [ ]:
from pathlib import Path

from gtsrb.config import load_config
from gtsrb.data import (
    build_dataloaders,
    estimate_channel_statistics,
    load_raw_split,
    read_labels,
    validate_dataset,
)
from gtsrb.data.validation import compare_statistics
from gtsrb.evaluation import plot_class_distribution, plot_samples_per_class

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
config = load_config([
    REPO_ROOT / 'configs' / 'data.yaml',
    REPO_ROOT / 'configs' / 'model.yaml',
    REPO_ROOT / 'configs' / 'train.yaml',
])
print('image size :', config.data.image_size)
print('num classes:', config.data.num_classes)
print('split seed :', config.data.split_seed)


## 1. Load the raw dataset

`torchvision`'s GTSRB `train` split holds **26,640** images, not the official training
partition's **39,209**. That caveat travels with every number in this project.


In [ ]:
train = load_raw_split(REPO_ROOT / config.data.root, 'train', download=False)
test = load_raw_split(REPO_ROOT / config.data.root, 'test', download=False)
print(f'train split: {len(train):,} images')
print(f'test split : {len(test):,} images')


## 2. Validate and describe

Structural checks — label range, image mode, aspect ratio, class distribution — and the
class-balance report. GTSRB is long-tailed, so min/max per class is the number that matters.


In [ ]:
stats = validate_dataset(train, num_classes=config.data.num_classes, sample_size=500, seed=config.data.split_seed)
print(stats.summary())
print('missing classes:', stats.missing_classes or 'none')


In [ ]:
counts = stats.class_counts
plot_class_distribution(counts, REPO_ROOT / 'notebooks' / 'class_distribution.png',
                        title='GTSRB training split — class distribution')
print('min/max per class:', stats.min_class_count, '/', stats.max_class_count,
      f'(imbalance {stats.imbalance_ratio:.1f}x)')


## 3. The split manifest

The split is computed once and persisted with its indices **and labels**, then verified on
every load. This cell shows the partition and the achieved class balance per split.


In [ ]:
bundle = build_dataloaders(config, download=False)
m = bundle.manifest
print(f'split seed {m.seed}, val_fraction {m.val_fraction}')
print(f'  train: {m.num_train:,}')
print(f'  val  : {m.num_val:,}')
print(f'  test : {len(bundle.test_dataset):,}')
assert m.num_train + m.num_val == m.num_samples
assert set(m.train_indices).isdisjoint(m.val_indices), 'the split leaks'
print('\ntrain/val are disjoint and cover the dataset exactly once')


In [ ]:
# What class-balanced sampling actually does, per class.
raw = m.class_counts('train')
effective = bundle.class_weights.effective_counts(bundle.train_dataset.labels, len(bundle.train_dataset))
rarest = min(raw, key=lambda c: raw[c])
commonest = max(raw, key=lambda c: raw[c])
print(f'rarest class {rarest}: {raw[rarest]} samples -> {effective[rarest]:.1f} expected draws/epoch')
print(f'commonest    {commonest}: {raw[commonest]} samples -> {effective[commonest]:.1f} expected draws/epoch')


## 4. Sample images

One image per class, with the count of correctly-predicted samples from the evaluated
checkpoint. Run `make evaluate` first for the accuracy annotation to be meaningful.


In [ ]:
import torch
images, targets = [], []
seen = set()
for i in range(len(bundle.test_dataset)):
    x, y = bundle.test_dataset[i]
    if int(y) in seen:
        continue
    seen.add(int(y))
    images.append(x)
    targets.append(y)
    if len(seen) == config.data.num_classes:
        break
plot_samples_per_class(torch.stack(images), torch.stack(targets),
                       REPO_ROOT / 'notebooks' / 'sample_per_class.png',
                       mean=config.data.normalize_mean, std=config.data.normalize_std,
                       title='One test image per class')
print(f'{len(images)} classes shown')


## 5. Normalisation statistics

The configured constants are re-measured from the data rather than trusted. On the real
64×64-resized split the configured mean matches a mean-of-per-image-means estimator to
within 0.002 and the std matches the pooled estimator to within 0.002.


In [ ]:
measured_mean, measured_std = estimate_channel_statistics(
    train, sample_size=2000, seed=config.data.split_seed, image_size=config.data.image_size
)
report = compare_statistics(measured_mean, measured_std,
                            config.data.normalize_mean, config.data.normalize_std)
print('measured (pixel-pooled):', [round(v, 4) for v in report['measured_mean']])
print('configured             :', report['reference_mean'])
print('mean delta             :', [round(v, 4) for v in report['mean_delta']])
print('std delta              :', [round(v, 4) for v in report['std_delta']])
print('within materiality     :', report['within_tolerance'])


## 6. What the original notebook did not record

The original computed its split inside a cell and never persisted it, so "we used the same
split" was unverifiable. The manifest this notebook just asserted against is the fix —
and it is verified against the dataset on every load, including a per-index label check
that catches a stale manifest from a previous dataset revision.
